In [ ]:
import pandas as pd
import numpy as np
from ChromaVDB.chroma import ChromaFramework
from DeepGraphDB import DeepGraphDB
from tqdm.notebook import tqdm
import torch
import pickle

gdb = DeepGraphDB()
gdb.load_graph("/home/cc/PHD/dglframework/DeepKG/DeepGraphDB/graphs/primekg.bin")

# vdb = ChromaFramework(persist_directory="./ChromaVDB/chroma_db")
# records = vdb.list_records()

# names = [record['name'] for record in records if record['embedding_type'] == 'graph']
# entities = [record['entity'] for record in records if record['embedding_type'] == 'graph']
# graph_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'graph']
# text_embs = [record['embeddings'] for record in records if record['embedding_type'] == 'text']
# ids = [record['id'] for record in records if record['embedding_type'] == 'graph']

data = pd.read_excel('data/2025_03_29.xlsx') # Provare ad usare anche stadio-avanzato, IPI e Log10hGE

with open("/home/cc/PHD/dglframework/DeepKG/conditioning.pkl", "rb") as f:
    conditioning = pickle.load(f)

target = 35884

In [ ]:
nodes_to_keep = {}
scores_dict = {}

conditioning.append({ 'entity': 'disease: diffuse large B-cell lymphoma', 'score': 5 })

for item in conditioning:
    entity = item['entity'].split(': ')
    score = item['score']

    if entity[1] in gdb.node_data[entity[0]]['name']:
        idx = np.where(gdb.node_data[entity[0]]['name'] == entity[1])[0][0]

        nodes_to_keep[entity[0]] = nodes_to_keep.get(entity[0], []) + [idx]
        scores_dict[entity[0]] = scores_dict.get(entity[0], []) + [score]
    else:
        print(f"Entity {entity[1]} not found in graph for type {entity[0]}")

for entity_type, scores in scores_dict.items():
    scores_tensor = torch.tensor(scores, dtype=torch.float32, device='cuda')

    normalized_scores = scores_tensor / 5.0
    scores_dict[entity_type] = normalized_scores

finetune_graph = gdb.graph.subgraph(nodes_to_keep)

gdb.graph = finetune_graph

In [ ]:
import torch
import dgl
import numpy as np
from collections import defaultdict
from sklearn.model_selection import train_test_split
from DeepGraphDB.gnns.heteroSAGEattn import AdvancedHeteroLinkPredictor, compute_loss

in_feats = {ntype: gdb.graph.nodes['disease'].data['x'].shape[1] for ntype in gdb.graph.ntypes}

# Choose multiple edge types for prediction
# target_etypes = [ctype for ctype in gdb.graph.canonical_etypes if (ctype[0] == "geneprotein" or ctype[2] in "geneprotein") and gdb.graph.num_edges(ctype) > 5000]
target_etypes = [ctype for ctype in gdb.graph.canonical_etypes if (ctype[0] == "geneprotein" or ctype[2] in "geneprotein")]

print(f"Target edge types for prediction: {target_etypes}")

hidden_feats = 512
out_feats = 512

model = AdvancedHeteroLinkPredictor(
    node_types=gdb.graph.ntypes,  # All node types in the graph
    edge_types=gdb.graph.etypes,  # All edge types for GNN layers
    canonical_etypes=gdb.graph.canonical_etypes,  # All canonical edge types,
    in_feats=in_feats,
    hidden_feats=hidden_feats,
    out_feats=out_feats,
    scores=scores_dict,  # Use scores for link prediction
    # scores=None,  
    num_layers=3,
    use_attention=True,
    predictor_type='mlp',
    target_etypes=target_etypes,  # Only target edge types for prediction
    gnn_type='sage'
)

print(f"Model created with {sum(p.numel() for p in model.parameters())} parameters")

embs = gdb.train_model(model, compute_loss, target_etypes, 'cuda', bs=1000000, num_epochs=130)

In [ ]:
# torch.save(model, "/home/cc/PHD/dglframework/DeepKG/models/model-SAGE-bcell-finetune.pt")

In [ ]:

for entity in gdb.graph.ntypes:
    embeddings_tensor = embs[entity].cpu()
    torch.save(embeddings_tensor, f"/home/cc/PHD/dglframework/DeepKG/finetune-embs/{entity}.pt")